# Импорты

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
import os
import random
from torch.cuda.amp import autocast, GradScaler
warnings.filterwarnings('ignore')

In [ ]:
def set_seed(seed=42):
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"✓ Seed set to {seed} (deterministic mode)")

# Метрика

In [ ]:
def wape_plus_rbias(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    s = y_true.sum()
    if s == 0:
        return 0.0
    return np.abs(y_pred - y_true).sum() / s + np.abs(y_pred.sum() / s - 1)

# Загрузка данных

In [ ]:
train = pd.read_parquet('train_team_track.parquet')
test = pd.read_parquet('test_team_track.parquet')
train['timestamp'] = pd.to_datetime(train['timestamp'])
test['timestamp'] = pd.to_datetime(test['timestamp'])

route_to_office = train.groupby('route_id')['office_from_id'].first().to_dict()
test['office_from_id'] = test['route_id'].map(route_to_office)

status_cols = [f'status_{i}' for i in range(1, 9)]
for c in status_cols:
    test[c] = np.nan

WINDOW_SIZE = 48

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed(seed)
set_seed()
        
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Конфигурации

In [ ]:
MODE = 'final'

configs = {
    'experiment': {
        'sample_routes': 200,
        'epochs': 12,
        'patience': 4,
        'batch_size': 2048
    },
    'validation': {
        'sample_routes': 500,
        'epochs': 20,
        'patience': 5,
        'batch_size': 2048
    },
    'final': {
        'sample_routes': None,
        'epochs': 40,
        'patience': 10,
        'batch_size': 2048
    }
}

cfg = configs[MODE]
print(f"MODE: {MODE}")
print(f"Config: {cfg}")

EPOCHS = cfg['epochs']
PATIENCE = cfg['patience']
BS = cfg['batch_size']

### Смесь

In [ ]:
if cfg['sample_routes']:
    sample_routes = np.random.choice(
        train['route_id'].unique(),
        size=cfg['sample_routes'],
        replace=False
    )
#     wrost_routes = [268, 675, 897, 733, 717, 519]
#     sample_routes[sample_routes.shape[0] - len(wrost_routes):] = wrost_routes
    train = train[train['route_id'].isin(sample_routes)].copy()
train = train.reset_index(drop=True)
# np.save('sample_routes_experiment.npy', sample_routes)

### Только плохие

In [ ]:
bad_routes = [
    243, 705, 385, 602, 823, 77, 939, 844, 406, 190, 629, 936, 43, 420, 555,
    31, 326, 386, 966, 124, 268, 455, 71, 370, 703, 790, 215, 221, 882, 9,
    316, 407, 897, 920, 577, 253, 977, 28, 248, 519, 498, 637, 307, 928, 319,
    539, 250, 328, 185, 874, 361, 364, 464, 838, 911, 106, 301, 557, 659, 48,
    324, 528, 471, 746, 383, 440, 454, 921, 429, 846, 431, 776, 280, 390, 668,
    775, 803, 829, 265, 244, 931, 134, 245, 359, 604, 603, 172, 613, 78, 672,
    125, 679, 860, 791, 505, 626, 21, 809, 877, 542, 81, 195, 436, 811, 38,
    350, 42, 709, 851, 338, 291, 300, 616, 617, 387, 793, 717, 40, 282, 147,
    728, 837, 16, 956, 45, 531, 739, 474, 270, 278, 367, 0, 635, 463, 126, 889,
    50, 638, 902, 136, 417, 768, 450, 595, 187, 845, 562, 413, 912, 704, 87,
    58, 320, 854, 968, 796, 959, 150, 574, 346, 546, 446, 712, 946, 170, 236,
    675, 738, 304, 678, 5, 896, 96, 377, 849, 461, 252, 556, 500, 644, 870,
    563, 517, 927, 131, 581, 850, 951, 590, 225, 682, 220, 381, 3, 382, 355,
    141, 373, 74, 322, 62, 480, 906, 437, 514, 591, 35, 560, 371, 80, 518, 884,
    162, 689, 869, 525, 875, 360, 495, 102, 426, 745, 727, 308, 687, 916, 69,
    733, 864, 601, 135, 881, 122, 651, 493, 609, 723, 151, 582, 685, 284, 70,
    848, 918, 862, 566, 569, 633, 969, 204, 647, 527, 267, 357, 526, 22, 188,
    262, 789, 397, 891, 901, 804, 841, 996, 598, 273, 388, 179, 639, 175, 506,
    286, 137, 46, 340
]

In [ ]:
# sample_routes = np.random.choice(
#     np.array(bad_routes),
#     size=cfg['sample_routes'],
#     replace=False
# )
train = train[train['route_id'].isin(bad_routes)].copy()
train = train.reset_index(drop=True)

### Плохие без невозможных

In [ ]:
bad_routes = [
    243, 705, 385, 602, 823, 77, 939, 844, 406, 190, 629, 936, 43, 420, 555,
    31, 326, 386, 966, 124, 268, 455, 71, 370, 703, 790, 215, 221, 882, 9,
    316, 407, 897, 920, 577, 253, 977, 28, 248, 519, 498, 637, 307, 928, 319,
    539, 250, 328, 185, 874, 361, 364, 464, 838, 911, 106, 301, 557, 659, 48,
    324, 528, 471, 746, 383, 440, 454, 921, 429, 846, 431, 776, 280, 390, 668,
    775, 803, 829, 265, 244, 931, 134, 245, 359, 604, 603, 172, 613, 78, 672,
    125, 679, 860, 791, 505, 626, 21, 809, 877, 542, 81, 195, 436, 811, 38,
    350, 42, 709, 851, 338, 291, 300, 616, 617, 387, 793, 717, 40, 282, 147,
    728, 837, 16, 956, 45, 531, 739, 474, 270, 278, 367, 0, 635, 463, 126, 889,
    50, 638, 902, 136, 417, 768, 450, 595, 187, 845, 562, 413, 912, 704, 87,
    58, 320, 854, 968, 796, 959, 150, 574, 346, 546, 446, 712, 946, 170, 236,
    675, 738, 304, 678, 5, 896, 96, 377, 849, 461, 252, 556, 500, 644, 870,
    563, 517, 927, 131, 581, 850, 951, 590, 225, 682, 220, 381, 3, 382, 355,
    141, 373, 74, 322, 62, 480, 906, 437, 514, 591, 35, 560, 371, 80, 518, 884,
    162, 689, 869, 525, 875, 360, 495, 102, 426, 745, 727, 308, 687, 916, 69,
    733, 864, 601, 135, 881, 122, 651, 493, 609, 723, 151, 582, 685, 284, 70,
    848, 918, 862, 566, 569, 633, 969, 204, 647, 527, 267, 357, 526, 22, 188,
    262, 789, 397, 891, 901, 804, 841, 996, 598, 273, 388, 179, 639, 175, 506,
    286, 137, 46, 340
]
very_bad_routes = [243, 124, 370, 215, 407, 977, 928, 364, 931, 505, 21, 531, 474, 450, 968, 582]

available_routes = np.setdiff1d(bad_routes, very_bad_routes)

# sample_routes = np.random.choice(
#     available_routes,
#     size=cfg['sample_routes'],
#     replace=False
# )
train = train[train['route_id'].isin(available_routes)].copy()
train = train.reset_index(drop=True)

### Без плохих

In [ ]:
if cfg['sample_routes']:
    exclude_routes = [
        243, 705, 385, 602, 823, 77, 939, 844, 406, 190, 629, 936, 43, 420, 555,
        31, 326, 386, 966, 124, 268, 455, 71, 370, 703, 790, 215, 221, 882, 9,
        316, 407, 897, 920, 577, 253, 977, 28, 248, 519, 498, 637, 307, 928, 319,
        539, 250, 328, 185, 874, 361, 364, 464, 838, 911, 106, 301, 557, 659, 48,
        324, 528, 471, 746, 383, 440, 454, 921, 429, 846, 431, 776, 280, 390, 668,
        775, 803, 829, 265, 244, 931, 134, 245, 359, 604, 603, 172, 613, 78, 672,
        125, 679, 860, 791, 505, 626, 21, 809, 877, 542, 81, 195, 436, 811, 38,
        350, 42, 709, 851, 338, 291, 300, 616, 617, 387, 793, 717, 40, 282, 147,
        728, 837, 16, 956, 45, 531, 739, 474, 270, 278, 367, 0, 635, 463, 126, 889,
        50, 638, 902, 136, 417, 768, 450, 595, 187, 845, 562, 413, 912, 704, 87,
        58, 320, 854, 968, 796, 959, 150, 574, 346, 546, 446, 712, 946, 170, 236,
        675, 738, 304, 678, 5, 896, 96, 377, 849, 461, 252, 556, 500, 644, 870,
        563, 517, 927, 131, 581, 850, 951, 590, 225, 682, 220, 381, 3, 382, 355,
        141, 373, 74, 322, 62, 480, 906, 437, 514, 591, 35, 560, 371, 80, 518, 884,
        162, 689, 869, 525, 875, 360, 495, 102, 426, 745, 727, 308, 687, 916, 69,
        733, 864, 601, 135, 881, 122, 651, 493, 609, 723, 151, 582, 685, 284, 70,
        848, 918, 862, 566, 569, 633, 969, 204, 647, 527, 267, 357, 526, 22, 188,
        262, 789, 397, 891, 901, 804, 841, 996, 598, 273, 388, 179, 639, 175, 506,
        286, 137, 46, 340
    ]

    available_routes = np.setdiff1d(train['route_id'].unique(), exclude_routes)
    
    sample_routes = np.random.choice(
        available_routes,
        size=min(cfg['sample_routes'], len(available_routes)),
        replace=False
    )
    
    train = train[train['route_id'].isin(sample_routes)].copy()
else:
    exclude_routes = [
        243, 705, 385, 602, 823, 77, 939, 844, 406, 190, 629, 936, 43, 420, 555,
        31, 326, 386, 966, 124, 268, 455, 71, 370, 703, 790, 215, 221, 882, 9,
        316, 407, 897, 920, 577, 253, 977, 28, 248, 519, 498, 637, 307, 928, 319,
        539, 250, 328, 185, 874, 361, 364, 464, 838, 911, 106, 301, 557, 659, 48,
        324, 528, 471, 746, 383, 440, 454, 921, 429, 846, 431, 776, 280, 390, 668,
        775, 803, 829, 265, 244, 931, 134, 245, 359, 604, 603, 172, 613, 78, 672,
        125, 679, 860, 791, 505, 626, 21, 809, 877, 542, 81, 195, 436, 811, 38,
        350, 42, 709, 851, 338, 291, 300, 616, 617, 387, 793, 717, 40, 282, 147,
        728, 837, 16, 956, 45, 531, 739, 474, 270, 278, 367, 0, 635, 463, 126, 889,
        50, 638, 902, 136, 417, 768, 450, 595, 187, 845, 562, 413, 912, 704, 87,
        58, 320, 854, 968, 796, 959, 150, 574, 346, 546, 446, 712, 946, 170, 236,
        675, 738, 304, 678, 5, 896, 96, 377, 849, 461, 252, 556, 500, 644, 870,
        563, 517, 927, 131, 581, 850, 951, 590, 225, 682, 220, 381, 3, 382, 355,
        141, 373, 74, 322, 62, 480, 906, 437, 514, 591, 35, 560, 371, 80, 518, 884,
        162, 689, 869, 525, 875, 360, 495, 102, 426, 745, 727, 308, 687, 916, 69,
        733, 864, 601, 135, 881, 122, 651, 493, 609, 723, 151, 582, 685, 284, 70,
        848, 918, 862, 566, 569, 633, 969, 204, 647, 527, 267, 357, 526, 22, 188,
        262, 789, 397, 891, 901, 804, 841, 996, 598, 273, 388, 179, 639, 175, 506,
        286, 137, 46, 340
    ]
    
    available_routes = np.setdiff1d(train['route_id'].unique(), exclude_routes)
    train = train[train['route_id'].isin(available_routes)].copy()
    
train = train.reset_index(drop=True)

### Худшие для текущей модели

In [ ]:
bad_routes_for_model = [889, 733, 519, 897, 717, 301, 31, 381, 556, 906, 500, 703, 920, 71, 727]

train = train[train['route_id'].isin(bad_routes_for_model)].copy()
train = train.reset_index(drop=True)

### Без худших для текущей модели

In [ ]:
bad_routes_for_model = [889, 733, 519, 897, 717, 301, 31, 381, 556, 906, 500, 703, 920, 71, 727]
available_routes = np.setdiff1d(train['route_id'].unique(), bad_routes_for_model)
    
sample_routes = np.random.choice(
    available_routes,
    size=min(cfg['sample_routes'], len(available_routes)),
    replace=False
)
train = train[train['route_id'].isin(sample_routes)].copy()
train = train.reset_index(drop=True)

### Загрузка созданного датасета

In [ ]:
if cfg['sample_routes']:
    sample_routes = np.load('sample_routes_experiment.npy')    
    train = train[train['route_id'].isin(sample_routes)].copy()
train = train.reset_index(drop=True)

# Скейлеры

In [ ]:
class LogRobustScaler:
    def __init__(self):
        self.medians = {}
        self.iqrs = {}
        self.use_log = {}

    def fit(self, df, columns, log_columns=None):
        log_columns = log_columns or []
        for col in columns:
            v = df[col].dropna().values.copy()
            if col in log_columns:
                v = np.log1p(np.clip(v, 0, None))
                self.use_log[col] = True
            else:
                self.use_log[col] = False
            q25, q75 = np.percentile(v, [25, 75])
            self.medians[col] = np.median(v)
            self.iqrs[col] = max(q75 - q25, 1e-4)
        return self

    def transform(self, values, col):
        v = np.array(values, dtype=float)
        if self.use_log.get(col, False):
            v = np.log1p(np.clip(v, 0, None))
        return (v - self.medians[col]) / self.iqrs[col]

    def inverse_transform(self, values, col):
        v = np.array(values, dtype=float)
        v = v * self.iqrs[col] + self.medians[col]
        if self.use_log.get(col, False):
            v = np.expm1(v)
        return v

In [ ]:
log_cols = status_cols
scaler = LogRobustScaler()
scaler.fit(train, log_cols, log_columns=log_cols)

target_scaler = LogRobustScaler()
target_scaler.fit(train, ['target_2h'])

# Создание фичей

In [ ]:
def add_time_features(df):
    df = df.copy()
    df['time_slot'] = df['timestamp'].dt.hour * 2 + df['timestamp'].dt.minute // 30
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['hour'] = df['timestamp'].dt.hour + df['timestamp'].dt.minute/60
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(float)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['slot_sin'] = np.sin(2 * np.pi * df['time_slot'] / 48)
    df['slot_cos'] = np.cos(2 * np.pi * df['time_slot'] / 48)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    return df

train = add_time_features(train)
test = add_time_features(test)

In [ ]:
time_features = ['hour_sin', 'hour_cos', 'slot_sin', 'slot_cos',
                 'dow_sin', 'dow_cos', 'is_weekend']

seq_cols = status_cols + ['target_2h']
num_seq_features = len(seq_cols) + len(time_features)

# Построение окон

### Дефолтное

In [ ]:
print("Building sequences efficiently...")

train_sorted = train

for c in seq_cols[:-1]:
    train_sorted[f'{c}_norm'] = scaler.transform(train_sorted[c].values, c)
train_sorted['target_2h_norm'] = target_scaler.transform(train_sorted['target_2h'].values, 'target_2h')

train_sorted['target_2h_prev'] = train_sorted.groupby('route_id', sort=False)['target_2h_norm'].shift(1)
train_sorted = train_sorted.dropna().reset_index(drop=True)

all_X = []
all_y = []
all_rids = []
all_oids = []
all_ts = []

norm_cols = [f'{c}_norm' for c in seq_cols[:-1]]
feature_matrix = train_sorted[norm_cols + time_features + ['target_2h_prev']].values

route_groups = train_sorted.groupby('route_id', sort=False)

for rid, group_indices in route_groups.groups.items():
    indices = group_indices.values
    n_rows = len(indices)
    
    if n_rows <= WINDOW_SIZE:
        continue
    
    route_features = feature_matrix[indices]
    route_target = train_sorted.loc[indices, 'target_2h_norm'].values
    route_ts = train_sorted.loc[indices, 'timestamp'].values
    route_oid = train_sorted.loc[indices[0], 'office_from_id']
    
    for i in range(WINDOW_SIZE, n_rows):
        all_X.append(route_features[i - WINDOW_SIZE:i])
        all_y.append(route_target[i])
        all_rids.append(rid)
        all_oids.append(route_oid)
        all_ts.append(route_ts[i])

all_X = np.array(all_X, dtype=np.float16)
all_y = np.array(all_y, dtype=np.float16)
all_rids = np.array(all_rids, dtype=np.int16)
all_oids = np.array(all_oids, dtype=np.int16)
all_ts = np.array(all_ts)

print(f"Sequences: {all_X.shape}")

# Датасеты и даталоэдеры

In [ ]:
last_10_timestamps = sorted(train['timestamp'].unique())[-10:]
last_10_timestamps_np = pd.to_datetime(last_10_timestamps).to_numpy()
print(f"\nValidation timestamps (last 10): {last_10_timestamps[0]} — {last_10_timestamps[-1]}")

val_mask = np.isin(all_ts, last_10_timestamps_np)
train_mask = ~val_mask

X_tr, y_tr = all_X[train_mask], all_y[train_mask]
r_tr, o_tr = all_rids[train_mask], all_oids[train_mask]
X_va, y_va = all_X[val_mask], all_y[val_mask]
r_va, o_va = all_rids[val_mask], all_oids[val_mask]

print(f"Train: {X_tr.shape[0]:,}, Val: {X_va.shape[0]:,}")

In [ ]:
class TSDataset(Dataset):
    def __init__(self, X, y, r, o):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.r = torch.LongTensor(r)
        self.o = torch.LongTensor(o)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.r[i], self.o[i], self.y[i]

train_loader = DataLoader(TSDataset(X_tr, y_tr, r_tr, o_tr),
                          batch_size=BS, shuffle=True, pin_memory=True)
val_loader = DataLoader(TSDataset(X_va, y_va, r_va, o_va),
                        batch_size=BS * 2, shuffle=False, pin_memory=True)

# Модель

In [ ]:
class DeliveryNet(nn.Module):
    def __init__(self, nf=16, nr=1000, no=54,
                 re=32, oe=16, ch=64, lh=128, ll=2, dr=0.2):
        super().__init__()
        self.r_emb = nn.Embedding(nr, re)
        self.o_emb = nn.Embedding(no, oe)

        self.cnn = nn.Sequential(
            nn.Conv1d(nf, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.GELU(),
            nn.Conv1d(ch, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.GELU(),
        )
        self.lstm = nn.LSTM(ch, lh, ll, batch_first=True,
                            dropout=dr if ll > 1 else 0)
        self.ln = nn.LayerNorm(lh)

        self.attn = nn.Sequential(
            nn.Linear(lh, lh // 2), nn.Tanh(), nn.Linear(lh // 2, 1))

        head_in = lh * 2 + re + oe
        self.head = nn.Sequential(
            nn.Linear(head_in, 128), nn.GELU(), nn.Dropout(dr),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(dr),
            nn.Linear(64, 1)
        )

    def forward(self, x, rid, oid):
        h = self.cnn(x.permute(0, 2, 1)).permute(0, 2, 1)
        h, _ = self.lstm(h)
        h = self.ln(h)

        w = torch.softmax(self.attn(h), dim=1)
        ctx = (h * w).sum(dim=1)

        last = h[:, -1, :]

        r = self.r_emb(rid)
        o = self.o_emb(oid)
        return self.head(torch.cat([ctx, last, r, o], dim=1)).squeeze(-1)

In [ ]:
model = DeliveryNet(nf=num_seq_features).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
model.load_state_dict(torch.load('model_weights_not_sparse_all.pth'))

# Функции ошибки

In [ ]:
class WeightedWapeBiasLoss(nn.Module):
    def __init__(self, bias_w=0.3):
        super().__init__()
        self.bias_w = bias_w
    
    def forward(self, pred, true):
        weights = torch.sqrt(torch.abs(true) + 1.0)
        weighted_mae = ((pred - true).abs() * weights).mean()
        bias = (pred.mean() - true.mean()).abs()
        return weighted_mae + self.bias_w * bias

criterion = WeightedWapeBiasLoss(0.3)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-5)

In [ ]:
best_metric, best_epoch, best_state = float('inf'), 0, None
checkpoint_path = 'best_model.pth'

# Обучение

### Дефолтная тренировка

In [ ]:
print("\nTraining...")
for ep in range(EPOCHS):
    model.train()
    losses = []
    for xb, rb, ob, yb in train_loader:
        xb, rb, ob, yb = [t.to(device) for t in [xb, rb, ob, yb]]
        
        if torch.isnan(xb).any() or torch.isinf(xb).any():
            print("NaN/Inf in input!")
            print(f"xb stats: min={xb.min()}, max={xb.max()}, mean={xb.mean()}")
            break
            
        optimizer.zero_grad()
        loss = criterion(model(xb, rb, ob), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()

    model.eval()
    vp, vt = [], []
    with torch.no_grad(), autocast():
        for xb, rb, ob, yb in val_loader:
            vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
            vt.append(yb.numpy())

    vp_d = np.clip(target_scaler.inverse_transform(np.concatenate(vp), 'target_2h'), 0, None)
    vt_d = target_scaler.inverse_transform(np.concatenate(vt), 'target_2h')
    m = wape_plus_rbias(vt_d, vp_d)

    if m < best_metric:
        best_metric, best_epoch = m, ep
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        torch.save({
            'epoch': ep,
            'model_state': best_state,
            'metric': best_metric,
            'optimizer_state': optimizer.state_dict(),
        }, checkpoint_path)
        print(f"  → Saved checkpoint: {checkpoint_path}")


    w = np.abs(vp_d - vt_d).sum() / vt_d.sum()
    b = np.abs(vp_d.sum() / vt_d.sum() - 1)
    print(f"Ep {ep:3d} | L {np.mean(losses):.4f} | "
          f"M {m:.4f} (W:{w:.4f} B:{b:.4f})"
          f"{' *' if m <= best_metric else ''}")

    if ep - best_epoch >= PATIENCE:
        print(f"Early stop at {ep}")
        break

In [ ]:
print(f"\nBest: ep {best_epoch}, metric {best_metric:.6f}")
model.load_state_dict(best_state)
model = model.to(device)

In [ ]:
torch.save(model.state_dict(), 'model_weights_sparse_all.pth')

# Предсказание статусов

## Модель

In [ ]:
class StatusPredictor(nn.Module):
    def __init__(self, window_size=48, n_statuses=8, n_time_features=7,
                 hidden_dim=64, lstm_hidden=96, num_layers=2, dropout=0.2):
        super().__init__()
        
        self.window_size = window_size
        
        self.status_lstm = nn.LSTM(
            n_statuses, lstm_hidden, num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0
        )
        self.status_ln = nn.LayerNorm(lstm_hidden)
        
        self.status_attn = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden // 2), nn.Tanh(),
            nn.Linear(lstm_hidden // 2, 1)
        )
        
        head_in = lstm_hidden * 2 + n_time_features
        
        self.head = nn.Sequential(
            nn.Linear(head_in, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
    
    def forward(self, status_history, time_features):
        h, _ = self.status_lstm(status_history)
        h = self.status_ln(h)
        
        w = torch.softmax(self.status_attn(h), dim=1)
        ctx = (h * w).sum(dim=1)
        
        last = h[:, -1, :]
        
        combined = torch.cat([ctx, last, time_features], dim=1)
        
        pred = torch.relu(self.head(combined))
        
        return pred.squeeze(-1)

## Подготовка данных

In [ ]:
def prepare_status_training_data(train_sorted, status_cols, time_features, 
                                  WINDOW_SIZE, last_10_timestamps_np):
    print("\n" + "="*70)
    print("PREPARING DATA FOR STATUS PREDICTION MODELS")
    print("="*70)
    
    status_train_data = []
    
    norm_cols_features = [f'{c}_norm' for c in status_cols]
    
    timestamp_in_val = set(last_10_timestamps_np)
    
    for rid in train_sorted['route_id'].unique():
        rd = train_sorted[train_sorted['route_id'] == rid]
        
        if len(rd) <= WINDOW_SIZE + 1:
            continue
        
        status_history_all = rd[norm_cols_features].values
        time_features_all = rd[time_features].values
        next_statuses_all = rd[status_cols].values  
        timestamps_all = rd['timestamp'].values
        
        for i in range(WINDOW_SIZE, len(rd) - 1):
            status_history = status_history_all[i - WINDOW_SIZE:i]
            time_features_current = time_features_all[i]
            next_statuses = next_statuses_all[i + 1]
            timestamp = timestamps_all[i]
            
            status_train_data.append({
                'route_id': rid,
                'timestamp': timestamp,
                'status_history': status_history,
                'time_features': time_features_current,
                'next_statuses': next_statuses
            })
    
    print(f"Created {len(status_train_data)} samples for status prediction")
    
    val_status_data = [s for s in status_train_data if s['timestamp'] in timestamp_in_val]
    train_status_data = [s for s in status_train_data if s['timestamp'] not in timestamp_in_val]
    
    print(f"Train: {len(train_status_data):,}, Val: {len(val_status_data):,}")
    
    return train_status_data, val_status_data

## Обучение

In [ ]:
from torch.utils.data import TensorDataset

def train_status_models(train_status_data, val_status_data, status_cols, 
                       time_features, device, epochs=15, batch_size=512):
    
    print("\nPreparing data for all status models...")
    
    X_train_history = np.array([s['status_history'] for s in train_status_data], dtype=np.float32)
    X_train_time = np.array([s['time_features'] for s in train_status_data], dtype=np.float32)
    y_train_all = np.array([s['next_statuses'] for s in train_status_data], dtype=np.float32)
    
    X_val_history = np.array([s['status_history'] for s in val_status_data], dtype=np.float32)
    X_val_time = np.array([s['time_features'] for s in val_status_data], dtype=np.float32)
    y_val_all = np.array([s['next_statuses'] for s in val_status_data], dtype=np.float32)
    
    y_train_stats = {}
    y_train_norm_all = np.zeros_like(y_train_all, dtype=np.float32)
    y_val_norm_all = np.zeros_like(y_val_all, dtype=np.float32)
    
    for status_idx in range(len(status_cols)):
        y_train = y_train_all[:, status_idx]
        y_train_mean = y_train.mean()
        y_train_std = y_train.std() + 1e-5
        
        y_train_stats[status_idx] = (y_train_mean, y_train_std)
        y_train_norm_all[:, status_idx] = (y_train - y_train_mean) / y_train_std
        y_val_norm_all[:, status_idx] = (y_val_all[:, status_idx] - y_train_mean) / y_train_std
    
    train_loaders = {}
    val_loaders = {}
    
    for status_idx, status_col in enumerate(status_cols):
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train_history),
            torch.FloatTensor(X_train_time),
            torch.FloatTensor(y_train_norm_all[:, status_idx])
        )
        val_dataset = TensorDataset(
            torch.FloatTensor(X_val_history),
            torch.FloatTensor(X_val_time),
            torch.FloatTensor(y_val_norm_all[:, status_idx])
        )
        
        train_loaders[status_idx] = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True, 
            pin_memory=True
        )
        val_loaders[status_idx] = DataLoader(
            val_dataset, batch_size=batch_size * 2, shuffle=False, 
            pin_memory=True
        )
    
    print("Data prepared")

    
    status_models = {}
    
    for status_idx, status_col in enumerate(status_cols):
        print(f"\n{'='*70}")
        print(f"Training model for {status_col} ({status_idx + 1}/8)")
        print(f"{'='*70}")
        
        train_loader = train_loaders[status_idx]
        val_loader = val_loaders[status_idx]
        
        model = StatusPredictor().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=5, T_mult=2, eta_min=1e-5
        )
        criterion = nn.SmoothL1Loss()
        
        best_val_loss = float('inf')
        best_state = None
        patience_counter = 0
        
        for ep in range(epochs):
            model.train()
            train_loss = 0
            
            for x_hist, x_time, y in train_loader:
                x_hist = x_hist.to(device)
                x_time = x_time.to(device)
                y = y.to(device)
                
                optimizer.zero_grad()
                pred = model(x_hist, x_time)
                loss = criterion(pred, y)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                train_loss += loss.item()
            
            scheduler.step()
            train_loss /= len(train_loader)
            
            model.eval()
            val_loss = 0
            
            with torch.no_grad():
                for x_hist, x_time, y in val_loader:
                    x_hist = x_hist.to(device)
                    x_time = x_time.to(device)
                    y = y.to(device)
                    
                    pred = model(x_hist, x_time)
                    loss = criterion(pred, y)
                    val_loss += loss.item()
            
            val_loss /= len(val_loader)
            
            is_best = val_loss < best_val_loss
            if is_best:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
                marker = " *"
            else:
                patience_counter += 1
                marker = ""
            
            if (ep + 1) % 2 == 0 or ep == 0:
                print(f"Ep {ep:2d} | Train L: {train_loss:.4f} | Val L: {val_loss:.4f}{marker}")
            
            if patience_counter >= 4:
                print(f"Early stop at epoch {ep}")
                break
        
        model.load_state_dict(best_state)
        model.eval()
        
        y_train_mean, y_train_std = y_train_stats[status_idx]
        
        status_models[status_col] = {
            'model': model,
            'mean': y_train_mean,
            'std': y_train_std
        }
        
        print(f"{status_col} trained. Best val loss: {best_val_loss:.4f}")
    
    return status_models

## Предсказание статусов

In [ ]:
def predict_next_statuses(route_histories, rid, status_cols, time_features, 
                         status_models, scaler, device, WINDOW_SIZE, row):
    
    h = route_histories[rid]
    h_array = np.array([arr[:8] for arr in h[-WINDOW_SIZE:]], dtype=np.float32)
    time_vals_array = np.array([row[f] for f in time_features], dtype=np.float32)
    
    predicted_statuses = []
    
    with torch.no_grad():
        for status_col in status_cols:
            status_model = status_models[status_col]['model']
            mean = status_models[status_col]['mean']
            std = status_models[status_col]['std']
            
            pred_norm = status_model(
                torch.FloatTensor(h_array).unsqueeze(0).to(device),
                torch.FloatTensor(time_vals_array).unsqueeze(0).to(device)
            ).cpu().numpy()[0]
            
            pred_denorm = pred_norm * std + mean
            pred_denorm = np.clip(pred_denorm, 0, None)
            
            predicted_statuses.append(pred_denorm)
    
    return np.array(predicted_statuses)

## В действии

In [ ]:
last_10_timestamps_np = pd.to_datetime(last_10_timestamps).to_numpy()

train_status_data, val_status_data = prepare_status_training_data(
    train_sorted, 
    status_cols, 
    time_features,
    WINDOW_SIZE,
    last_10_timestamps_np
)

In [ ]:
print("\n" + "="*70)
print("TRAINING STATUS PREDICTION MODELS")
print("="*70)

status_models = train_status_models(
    train_status_data, 
    val_status_data, 
    status_cols, 
    time_features, 
    device,
    epochs=15,
    batch_size=512
)

print("\nAll status models trained!")

In [ ]:
import joblib

In [ ]:
joblib.dump(status_models, 'status_models_sparse.joblib')

In [ ]:
for i, status in enumerate(status_cols):
    torch.save(status_models[status]['model'].state_dict(), f'model_weights_sparse_status{i+1}.pth')

In [ ]:
status_models = joblib.load('status_models.joblib')

In [ ]:
train.hour = train.timestamp.dt.hour + train.timestamp.dt.minute/60
train.hour

# Предсказание тестовых данных

### Дефолтное

In [ ]:
test = test[test['route_id'].isin(available_routes)]

In [ ]:
test = train_sorted.groupby('route_id').tail(10)

In [ ]:
test = test.reset_index().rename(columns={'index':'id'})

In [ ]:
train_sorted = train_sorted[train_sorted.groupby('route_id', sort=False).cumcount().between(4341-WINDOW_SIZE-10, 4341-11)]

In [ ]:
last_values = train_sorted.groupby('route_id', sort=False)['target_2h_norm'].last().to_dict()

test['target_2h'] = np.nan
test['target_2h_prev'] = np.nan
test['target_2h_norm'] = np.nan

test.loc[test.groupby('route_id', sort=False).head(1).index, 'target_2h_prev'] = test['route_id'].map(last_values)

In [ ]:
route_histories = {}
route_raw_statuses = {}
route_raw_target = {}

for rid in train_sorted['route_id'].unique():
    rd = train_sorted[train_sorted['route_id'] == rid]
    norm_parts = [scaler.transform(rd[c].values, c) for c in status_cols]
    norm_arr = np.column_stack(norm_parts)
    time_arr = rd[time_features].values
    target_arr = rd['target_2h_prev'].values.reshape(-1, 1)
    full = np.column_stack([norm_arr, time_arr, target_arr]).astype(np.float32)
    route_histories[rid] = list(full)
    route_raw_statuses[rid] = rd[status_cols].values.tolist()
    route_raw_target[rid] = rd['target_2h'].values.tolist()

test_sorted = test
test_timestamps = sorted(test['timestamp'].unique())

In [ ]:
id_to_pred = {}

model.eval()

for step, ts in enumerate(test_timestamps):
    rids = test_sorted[test_sorted['timestamp'] == ts]['route_id'].values.astype(int)
    oids = test_sorted[test_sorted['timestamp'] == ts]['office_from_id'].values.astype(int)
    batch_ids = test_sorted[test_sorted['timestamp'] == ts]['id'].values

    batch_X = []
    for rid in rids:
        h = route_histories[rid]
        w = np.array(h[-WINDOW_SIZE:], dtype=np.float32)
        batch_X.append(w)

    batch_X = np.nan_to_num(np.array(batch_X), nan=0.0)

    with torch.no_grad():
        preds_norm = model(
            torch.FloatTensor(batch_X).to(device),
            torch.LongTensor(rids).to(device),
            torch.LongTensor(oids).to(device)
        ).cpu().numpy()

    preds = np.clip(target_scaler.inverse_transform(preds_norm, 'target_2h'), 0, None)

    test_sorted.loc[test_sorted.groupby('route_id', sort=False).cumcount() == step, 'target_2h'] = preds
    if (step != 9):
        test_sorted.loc[test_sorted.groupby('route_id', sort=False).cumcount() == step+1, 'target_2h_prev'] = preds_norm
    test_sorted.loc[test_sorted.groupby('route_id', sort=False).cumcount() == step, 'target_2h_norm'] = preds_norm
    
    batch_df = test_sorted[test_sorted['timestamp'] == ts]
    
    for j, (_, row) in enumerate(batch_df.iterrows()):
        rid = int(row['route_id'])
        original_id = row['id']

        id_to_pred[original_id] = preds[j]

        raw_s = route_raw_statuses[rid]
        approx_s = np.mean(raw_s[-4:], axis=0)
        
        norm_s = [scaler.transform(np.array([approx_s[k]]), status_cols[k])[0]
                  for k in range(8)]
        t_vals = [row[f] for f in time_features]

        norm_t = [row['target_2h_prev']]
        
        new_step = np.array(norm_s + t_vals + norm_t, dtype=np.float32)
        route_histories[rid].append(new_step)
        route_raw_statuses[rid].append(approx_s.tolist())
        route_raw_target[rid].append(preds[j])


In [ ]:
submission = pd.DataFrame({
    'id': list(id_to_pred.keys()),
    'y_pred': list(id_to_pred.values())
})
submission = submission.sort_values('id').reset_index(drop=True)

In [ ]:
submission.to_csv('submission_nn_sparse.csv', index=False)
print(f"\nDone! {submission.shape}")
print(f"Sum: {submission['y_pred'].sum():.0f}")
print(submission.head(10))

In [ ]:
model.eval()
vp, vt = [], []
with torch.no_grad(), autocast():
    for xb, rb, ob, yb in val_loader:
        vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
        vt.append(yb.numpy())

vp_d = np.clip(target_scaler.inverse_transform(np.concatenate(vp), 'target_2h'), 0, None)
vt_d = target_scaler.inverse_transform(np.concatenate(vt), 'target_2h')
wape_plus_rbias(vt_d, vp_d)

### Со статусами

In [ ]:
last_values = train_sorted.groupby('route_id', sort=False)['target_2h_norm'].last().to_dict()

test['target_2h'] = np.nan
test['target_2h_prev'] = np.nan
test['target_2h_norm'] = np.nan

test.loc[test.groupby('route_id', sort=False).head(1).index, 'target_2h_prev'] = test['route_id'].map(last_values)

In [ ]:
route_histories = {}
route_raw_statuses = {}
route_raw_target = {}

for rid in train_sorted['route_id'].unique():
    rd = train_sorted[train_sorted['route_id'] == rid]
    norm_parts = [scaler.transform(rd[c].values, c) for c in status_cols]
    norm_arr = np.column_stack(norm_parts)
    time_arr = rd[time_features].values
    target_arr = rd['target_2h_prev'].values.reshape(-1, 1)
    full = np.column_stack([norm_arr, time_arr, target_arr]).astype(np.float32)
    route_histories[rid] = list(full)
    route_raw_statuses[rid] = rd[status_cols].values.tolist()
    route_raw_target[rid] = rd['target_2h'].values.tolist()

test_sorted = test
test_timestamps = sorted(test['timestamp'].unique())

In [ ]:
id_to_pred = {}

model.eval()

for status_col in status_cols:
    status_models[status_col]['model'].eval()

for step, ts in enumerate(test_timestamps):
    print(f"  Step {step+1}/10: {ts}")
    
    rids = test_sorted[test_sorted['timestamp'] == ts]['route_id'].values.astype(int)
    oids = test_sorted[test_sorted['timestamp'] == ts]['office_from_id'].values.astype(int)
    batch_ids = test_sorted[test_sorted['timestamp'] == ts]['id'].values

    batch_X = []
    for rid in rids:
        h = route_histories[rid]
        w = np.array(h[-WINDOW_SIZE:], dtype=np.float32)
        batch_X.append(w)

    batch_X = np.nan_to_num(np.array(batch_X), nan=0.0)

    with torch.no_grad():
        preds_norm = model(
            torch.FloatTensor(batch_X).to(device),
            torch.LongTensor(rids).to(device),
            torch.LongTensor(oids).to(device)
        ).cpu().numpy()

    preds = np.clip(target_scaler.inverse_transform(preds_norm, 'target_2h'), 0, None)

    test_sorted.loc[test_sorted.groupby('route_id', sort=False).cumcount() == step, 'target_2h'] = preds
    if (step != 9):
        test_sorted.loc[test_sorted.groupby('route_id', sort=False).cumcount() == step+1, 'target_2h_prev'] = preds_norm
    test_sorted.loc[test_sorted.groupby('route_id', sort=False).cumcount() == step, 'target_2h_norm'] = preds_norm
    
    batch_df = test_sorted[test_sorted['timestamp'] == ts]
    
    for j, (_, row) in enumerate(batch_df.iterrows()):
        rid = int(row['route_id'])
        original_id = row['id']

        id_to_pred[original_id] = preds[j]
        
        predicted_statuses = predict_next_statuses(
            route_histories, rid, status_cols, time_features,
            status_models, scaler, device, WINDOW_SIZE, row
        )
        
        norm_s = [scaler.transform(np.array([predicted_statuses[k]]), status_cols[k])[0]
                  for k in range(8)]
        t_vals = [row[f] for f in time_features]
        norm_t = [row['target_2h_prev']]
        
        new_step = np.array(norm_s + t_vals + norm_t, dtype=np.float32)
        route_histories[rid].append(new_step)
        route_raw_statuses[rid].append(predicted_statuses.tolist())
        route_raw_target[rid].append(preds[j])

In [ ]:
submission = pd.DataFrame({
    'id': list(id_to_pred.keys()),
    'y_pred': list(id_to_pred.values())
})
submission = submission.sort_values('id').reset_index(drop=True)

In [ ]:
submission.to_csv('submission_with_status_forecasting_sparse.csv', index=False)
print(f"Sum: {submission['y_pred'].sum():.0f}")
print(submission.head(10))

# Диагностика проблемы

### Дефолтная

In [ ]:
model.eval()
vp, vt = [], []
with torch.no_grad():
    for xb, rb, ob, yb in val_loader:
        vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
        vt.append(yb.numpy())

vp_d = np.clip(target_scaler.inverse_transform(np.concatenate(vp), 'target_2h'), 0, None)
vt_d = target_scaler.inverse_transform(np.concatenate(vt), 'target_2h')

errors = np.abs(vp_d - vt_d)
rel_errors = errors / (vt_d + 1e-5)

print("Error stats:")
print(f"  MAE: {errors.mean():.4f}")
print(f"  RMSE: {np.sqrt((errors**2).mean()):.4f}")
print(f"  Max error: {errors.max():.4f}")
print(f"  Median error: {np.median(errors):.4f}")
print(f"  90th percentile: {np.percentile(errors, 90):.4f}")
print(f"  % with error > 50: {(errors > 50).mean() * 100:.1f}%")

In [ ]:
val_df = pd.DataFrame({
    'route_id': r_va,
    'true': vt_d,
    'pred': vp_d
})
route_errors = val_df.groupby('route_id').apply(
    lambda g: np.abs(g['pred'] - g['true']).mean()
).sort_values(ascending=False)
print("\nWorst 10 routes:")
print(route_errors.head(10))

In [ ]:
train.groupby('route_id', sort=False)[['status_1', 'status_3']].mean().query('status_1 < 0.5 and status_3 < 0.5').index.tolist()

In [ ]:
val_df['error'] = np.abs(val_df['pred'] - val_df['true'])
val_df['step'] = val_df.groupby('route_id').cumcount()

error_by_step = val_df.groupby('step')['error'].agg(['mean', 'median', 'std'])
print("\nError by prediction step:")
print(error_by_step)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(error_by_step.index, error_by_step['mean'], marker='o', label='Mean')
plt.plot(error_by_step.index, error_by_step['median'], marker='s', label='Median')
plt.xlabel('Step (0=first, 9=last)')
plt.ylabel('Absolute Error')
plt.title('Error vs Prediction Step')
plt.legend()
plt.grid()
plt.show()

In [ ]:
worst_routes = [268, 889, 675, 897, 733, 717, 301, 519, 777, 31]

for rid in worst_routes[:3]:
    route_val = val_df[val_df['route_id'] == rid]
    print(f"\nRoute {rid} (MAE={route_val['error'].mean():.1f}):")
#     print(f"  Train samples: {len(train[train['route_id'] == rid])}")
#     print(f"  Val samples:   {len(route_val)}")
    print(f"  Mean target:   {route_val['true'].mean():.1f}")
    print(f"  Mean pred:     {route_val['pred'].mean():.1f}")
    print(f"  Target range:  {route_val['true'].min():.1f} - {route_val['true'].max():.1f}")
    
#     route_train = train[train['route_id'] == rid].copy()
#     route_train['time_slot'] = route_train['timestamp'].dt.hour * 2 + route_train['timestamp'].dt.minute // 30
#     profile = route_train.groupby('time_slot')['target_2h'].agg(['mean', 'std'])
#     print(f"  Target variability (CV): {(profile['std'] / profile['mean']).mean():.2f}")

In [ ]:
problematic = [268, 889, 300, 675]
random_routes = [605, 743, 540] 

for route_list, label in [(problematic, "PROBLEMATIC"), (random_routes, "RANDOM")]:
    print(f"\n{label} ROUTES:")
    for rid in route_list:
        if rid in sample_routes:
            route_data = train[train['route_id'] == rid]
            print(f"\nRoute {rid}:")
            print(f"  Samples: {len(route_data)}")
            print(f"  Target - Mean: {route_data['target_2h'].mean():.1f}, "
                  f"Std: {route_data['target_2h'].std():.1f}, "
                  f"Min: {route_data['target_2h'].min():.1f}, "
                  f"Max: {route_data['target_2h'].max():.1f}")
            print(f"  CoV: {route_data['target_2h'].std() / route_data['target_2h'].mean():.2f}")
            
            hourly = route_data.groupby('hour')['target_2h'].agg(['count', 'mean', 'std'])
            print(f"  Hourly pattern (count/mean/std):")
            print(hourly.to_string())
            
            for status_col in status_cols:
                print(f"  {status_col} - Mean: {route_data[status_col].mean():.1f}, "
                      f"Std: {route_data[status_col].std():.1f}")

In [ ]:
sparse_routes_list = list(sample_routes)[:20]

for rid in sparse_routes_list:
    rd = train[train['route_id'] == rid]
    print(f"\n=== Route {rid} ===")
    print(f"Samples: {len(rd)}")
    print(f"Target mean: {rd['target_2h'].mean():.2f}, std: {rd['target_2h'].std():.2f}")
    print(f"Target min: {rd['target_2h'].min():.2f}, max: {rd['target_2h'].max():.2f}")
    print(f"Office: {rd['office_from_id'].iloc[0]}")
    print("\nStatus stats:")
    for col in status_cols:
        print(f"  {col}: mean={rd[col].mean():.1f}, std={rd[col].std():.1f}, "
              f"min={rd[col].min():.1f}, max={rd[col].max():.1f}")
    
    print("\nCorrelation with target:")
    for col in status_cols:
        corr = rd[[col, 'target_2h']].corr().iloc[0, 1]
        print(f"  {col}: {corr:.3f}")

In [ ]:
sparse_train = train[train['route_id'].isin(sample_routes)]

print("Status usage in sparse routes:")
for col in status_cols:
    non_zero_pct = (sparse_train[col] > 0).mean() * 100
    mean_val = sparse_train[col].mean()
    print(f"{col}: {non_zero_pct:.1f}% non-zero, mean={mean_val:.1f}")

print("\nCorrelation with target (sparse routes only):")
corr_matrix = sparse_train[status_cols + ['target_2h']].corr()
print(corr_matrix['target_2h'].sort_values(ascending=False))

In [ ]:
train_sorted.route_id.unique()

In [ ]:
bad_routes = available_routes

In [ ]:
sparse_train = train_sorted[train_sorted['route_id'].isin(bad_routes)]

model.eval()
vp, vt = [], []
with torch.no_grad(), autocast():
    for xb, rb, ob, yb in val_loader:
        vp.append(model(xb.to(device), rb.to(device), ob.to(device)).cpu().numpy())
        vt.append(yb.numpy())

vp_d = np.clip(target_scaler.inverse_transform(np.concatenate(vp), 'target_2h'), 0, None)
vt_d = target_scaler.inverse_transform(np.concatenate(vt), 'target_2h')

errors = np.abs(vp_d - vt_d)

print("="*70)
print("MODEL_A ERRORS ON SPARSE ROUTES")
print("="*70)

print(f"\nOverall statistics:")
print(f"  MAE: {errors.mean():.2f}")
print(f"  RMSE: {np.sqrt((errors**2).mean()):.2f}")
print(f"  Median error: {np.median(errors):.2f}")
print(f"  Std error: {errors.std():.2f}")
print(f"  Max error: {errors.max():.2f}")


print(f"\nWorst 15 routes by MAE:")
val_df = pd.DataFrame({
    'route_id': r_va,
    'true': vt_d,
    'pred': vp_d
})
route_errors = val_df.groupby('route_id').apply(
    lambda g: np.abs(g['pred'] - g['true']).mean()
).sort_values(ascending=False)
print(route_errors.head(15))

for rid, mae in route_errors.reset_index().values:
    samples = len(sparse_train[sparse_train['route_id']==rid])
    mean_target = val_df[val_df.route_id ==rid]['true'].mean()
    print(f"  Route {rid}: MAE={mae:.2f}, target_mean={mean_target:.2f}, samples={samples}")

print(f"\nError distribution:")
print(f"  % errors < 10: {(errors < 10).mean() * 100:.1f}%")
print(f"  % errors < 20: {(errors < 20).mean() * 100:.1f}%")
print(f"  % errors < 50: {(errors < 50).mean() * 100:.1f}%")
print(f"  % errors >= 50: {(errors >= 50).mean() * 100:.1f}%")

print(f"\nError by target value:")
for threshold in [10, 30, 50, 100, 200]:
    mask = vt_d < threshold
    if mask.sum() > 0:
        print(f"  target < {threshold}: MAE={errors[mask].mean():.2f} ({mask.sum()} samples)")

bias = (vp_d - vt_d).mean()
print(f"\nBias analysis:")
print(f"  Mean bias: {bias:+.2f}")
print(f"  Model tends to {'OVERPREDICT' if bias > 0 else 'UNDERPREDICT'}")

print(f"\nPredictions statistics:")
print(f"  Mean pred: {vp_d.mean():.2f}, Mean true: {vt_d.mean():.2f}")
print(f"  Std pred: {vp_d.std():.2f}, Std true: {vt_d.std():.2f}")
print(f"  Min pred: {vp_d.min():.2f}, Min true: {vt_d.min():.2f}")
print(f"  Max pred: {vp_d.max():.2f}, Max true: {vt_d.max():.2f}")

print("="*70)

In [ ]:
df_normal = pd.read_csv('submission_with_status_forecasting_not_sparse.csv')
df_sparse = pd.read_csv('submission_with_status_forecasting_sparse.csv')
df_ultra_sparse = pd.read_csv('submission_nn_ultra_sparse.csv')

In [ ]:
res = pd.concat([df_normal, df_sparse, df_ultra_sparse], ignore_index=True)
res = res.sort_values('id').reset_index(drop=True)

In [ ]:
res

In [ ]:
res.to_csv('submission_nn7.csv', index=False)
print(f"\nDone! {res.shape}")
print(f"Sum: {res['y_pred'].sum():.0f}")
print(res.head(10))